# Flight Ticket Status Challenge

## Problem Description
We need to determine the **current status of flight tickets** for each passenger.  

- If a passenger books a ticket and the flight still has empty seats, the ticket is **Confirmed**.  
- If the flight is already at full capacity, the passenger is placed on the **Waitlist**.  
- Return the result ordered by `passenger_id` in ascending order.

---

## Schema

### Table: Flights
| Column Name | Type | Description                  |
|-------------|------|------------------------------|
| flight_id   | INT  | Unique flight identifier      |
| capacity    | INT  | Maximum number of passengers |

### Table: Passengers
| Column Name  | Type     | Description                          |
|--------------|----------|--------------------------------------|
| passenger_id | INT      | Unique passenger identifier          |
| flight_id    | INT      | Flight booked by the passenger       |
| booking_time | DATETIME | Timestamp when the booking was made  |

---

## Sample Data

### Flights
| flight_id | capacity |
|-----------|----------|
| 1         | 2        |
| 2         | 2        |
| 3         | 1        |

### Passengers
| passenger_id | flight_id | booking_time        |
|--------------|-----------|---------------------|
| 101          | 1         | 2023-07-10 16:30:00 |
| 102          | 1         | 2023-07-10 17:45:00 |
| 103          | 1         | 2023-07-10 12:00:00 |
| 104          | 2         | 2023-07-05 13:23:00 |
| 105          | 2         | 2023-07-05 09:00:00 |
| 106          | 3         | 2023-07-08 11:10:00 |
| 107          | 3         | 2023-07-08 09:10:00 |

---

## Expected Output
| passenger_id | Status    |
|--------------|-----------|
| 101          | Confirmed | 
| 102          | Waitlist  | 
| 103          | Confirmed | 
| 104          | Confirmed | 
| 105          | Confirmed | 
| 106          | Waitlist  | 
| 107          | Confirmed |

---

## PySpark Code: Create DataFrames and Temp Views

```python


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType
from datetime import datetime

# Schema for Flights
flights_schema = StructType([
    StructField("flight_id", IntegerType(), False),
    StructField("capacity", IntegerType(), False)
])

# Schema for Passengers
passengers_schema = StructType([
    StructField("passenger_id", IntegerType(), False),
    StructField("flight_id", IntegerType(), False),
    StructField("booking_time", TimestampType(), False)
])

# Data for Flights
flights_data = [
    (1, 2),
    (2, 2),
    (3, 1)
]

# Data for Passengers
passengers_data = [
    (101, 1, datetime(2023, 7, 10, 16, 30, 0)),
    (102, 1, datetime(2023, 7, 10, 17, 45, 0)),
    (103, 1, datetime(2023, 7, 10, 12, 0, 0)),
    (104, 2, datetime(2023, 7, 5, 13, 23, 0)),
    (105, 2, datetime(2023, 7, 5, 9, 0, 0)),
    (106, 3, datetime(2023, 7, 8, 11, 10, 0)),
    (107, 3, datetime(2023, 7, 8, 9, 10, 0))
]

# Create DataFrames
flights_df = spark.createDataFrame(flights_data, flights_schema)
passengers_df = spark.createDataFrame(passengers_data, passengers_schema)

# Register Temp Views
flights_df.createOrReplaceTempView("Flights")
passengers_df.createOrReplaceTempView("Passengers")

# Quick check
flights_df.show()
passengers_df.show()


In [0]:
%sql
SELECT date_format(booking_time, 'yyyy-MM-dd HH:mm:ss') AS booking_time
FROM Passengers;


In [0]:
%sql
with cte as (
    Select
        row_number()over(partition by p.flight_id order by p.booking_time asc) as rn
        , p.passenger_id  as passenger_id
        , p.flight_id
        , p.booking_time
        ,f.capacity as capacity
      from Flights f inner Join 
      Passengers p
       on f.flight_id = p.flight_id

        )
  select passenger_id ,-- , rn , capacity , flight_id,booking_time ,
  case 
    when rn <= capacity then 'Confirmed'
    else 'Waitlist'
    end as Status 
  from cte order by passenger_id asc
